In [1]:
# Time utilities for delays and execution timing
import time

# JSON serialization and deserialization
import json

# Tabular data manipulation and analysis
import pandas as pd

# Datetime utilities with explicit UTC handling
from datetime import datetime, timedelta, UTC

# YouTube Data API client builder
from googleapiclient.discovery import build

# YouTube API error handling
from googleapiclient.errors import HttpError

# Regular expressions for parsing and normalization
import re

# SQLite database interface
import sqlite3

# Filesystem path utilities
from pathlib import Path

# Normalize a keyword into a compact identifier.
# This helper lowercases the input and removes all
# spaces and hyphens to produce a single-token string.
# Input : keyword (str)
# Output: str   compact normalized keyword

def normalize_keyword(keyword: str) -> str:
    return (keyword.lower().strip().replace(" ", "").replace("-", ""))

## Youtube

In [2]:
# Loads YouTube Data API keys from a text file and validates them.
# This function is used to enable API key rotation in order to
# handle quota limits and ensure fault-tolerant data acquisition.
# Input : path (str)  path to a text file containing one API key per line
# Output: list[str]   cleaned list of YouTube API keys

def load_youtube_api_keys(path):
    # Open the file containing API keys
    with open(path, "r") as f:
        # Strip whitespace and remove empty lines
        keys = [k.strip() for k in f if k.strip()]
    # If no valid keys are found, stop execution
    if not keys:
        raise ValueError("No YouTube API keys found")
    # Log the number of loaded keys
    print(f" Loaded {len(keys)} YouTube API keys")
    # Return the list of API keys
    return keys


# Normalizes textual fields used in keyword matching.
# This function standardizes text by lowercasing and collapsing
# whitespace, enabling consistent comparisons across titles, tags,
# and keyword variants.
# Input : text (str)
# Output: str  normalized text

def normalize_text(text):
    # Replace multiple whitespace characters with a single space,
    # convert to lowercase and trim leading/trailing spaces
    return re.sub(r"\s+", " ", text.lower().strip())


# Generates multiple textual variants of a keyword to increase
# recall during YouTube search queries.
# Variants account for different formatting conventions commonly
# found in video titles and tags.
# Input : keyword (str)
# Output: list[str]  list of keyword variants

def generate_keyword_variants(keyword):
    # Normalize the input keyword
    base = normalize_text(keyword)
    # Split the keyword into individual words
    parts = base.split()
    # If the keyword has fewer than 2 words, return it as-is
    if len(parts) < 2:
        return [base]
    # Take the first two words only
    w1, w2 = parts[:2]
    # Generate common formatting variants
    return [f"{w1} {w2}",f"{w1}{w2}",f"{w1}_{w2}",f"{w1}-{w2}",f"{w1} - {w2}"]


# Splits a time interval into contiguous sub-windows of fixed length.
# This function is used to control temporal coverage and mitigate
# ranking instability and pagination limits in the YouTube API.
# Input : start_date (datetime), end_date (datetime), window_days (int)
# Output: list[tuple(datetime, datetime)]     list of time windows

def generate_time_windows(start_date, end_date, window_days):
    # Initialize the list of windows
    windows = []
    # Start from the initial date
    cur = start_date
    # Continue until the end date is reached
    while cur < end_date:
        # Define the next window boundary
        nxt = min(cur + timedelta(days=window_days), end_date)
        # Append the (start, end) tuple
        windows.append((cur, nxt))
        # Move to the next window
        cur = nxt
    # Return all generated windows
    return windows



# Splits a list into smaller chunks of fixed size.
# This helper is required to comply with YouTube API constraints,
# which limit the number of video IDs per request.
# Input : lst (list), size (int)
# Output: generator[list]  batches of list elements

def chunk_list(lst, size=50):
    # Iterate over the list in steps of `size`
    for i in range(0, len(lst), size):
        # Yield a sublist of length `size`
        yield lst[i:i + size]


# Parses an ISO 8601 duration string returned by the YouTube API
# and converts it into total seconds.
# This allows numerical analysis of video length and the
# identification of short-form content such as reels or shorts.
# Input : duration (str | None)
# Output: int | None               duration expressed in seconds


def parse_iso8601_duration(duration):
    # If duration is missing, return None
    if duration is None:
        return None
    # Regex pattern to extract hours, minutes, seconds
    pattern = re.compile(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?')
    # Apply the regex
    m = pattern.match(duration)
    # If the pattern does not match, return None
    if not m:
        return None
    # Extract hours, minutes and seconds with default 0
    h = int(m.group(1) or 0)
    m_ = int(m.group(2) or 0)
    s = int(m.group(3) or 0)
    # Convert everything to seconds
    return h * 3600 + m_ * 60 + s

# Constructs a standardized output file path for pipeline artifacts.
# This helper ensures consistent naming conventions across outputs
# and guarantees that the target directory exists before writing.
# Input : topic (str)        topic identifier used as filename prefix
#         suffix (str)       logical descriptor of the output content
#         output_dir (str)   base directory for output files
#         ext (str)          file extension (default: ndjson)
# Output: str                full path to the output file

def make_output_name(topic, suffix, output_dir, ext="ndjson"):
    # Convert output directory to a Path object
    output_dir = Path(output_dir)

    # Create the output directory if it does not already exist
    output_dir.mkdir(parents=True, exist_ok=True)

    # Build the output filename using the standard convention
    fname = f"{topic}_{suffix}.{ext}"

    # Return the full path as a string
    return str(output_dir / fname)

In [9]:
# VIDEO DISCOVERY 
# Function: youtube_search_videos
# Performs keyword-based video discovery on YouTube by searching across titles, descriptions and tags within
# a specified time range.
# The function systematically explores multiple keyword variants, temporal windows and ranking orders,
# while handling API quota limitations through API key rotation and in-memory resume logic.
# Writes raw discovery events to a newline-delimited JSON (NDJSON) file.
#
# Input :
#   - api_keys (list[str])            available YouTube API keys
#   - keyword (str)                   main topic keyword
#   - start_date (datetime)           start of the temporal search window
#   - end_date (datetime)             end of the temporal search window
#   - window_days (int)               size of each temporal sub-window
#   - max_pages_per_window (int)      pagination limit per window/order
#   - sleep_time (int)                delay between API requests
#   - output_json (str)               output NDJSON file path
#   - max_retries (int)               retry attempts for transient failures
#
# Output:
#   - list[dict] → collected discovery events
#   - int        → completion flag (1 = completed, 0 = quota interrupted)

def youtube_search_videos(api_keys, keyword, start_date, end_date, window_days=14, max_pages_per_window=10,
                          sleep_time=2, output_json=None, max_retries=3):

    # Enforce explicit output path to avoid accidental data loss
    if output_json is None:
        raise ValueError("output_json must be provided explicitly")

    # Normalize main keyword and generate recall-expanding variants
    keyword_norm = normalize_text(keyword)
    variants = generate_keyword_variants(keyword)

    # Supported YouTube API ordering strategies
    orders = ["viewCount", "relevance", "date"]

    # Split the global temporal range into smaller windows
    windows = generate_time_windows(start_date, end_date, window_days)

    # Container for collected discovery events
    events = []

    # Resume state used when rotating API keys due to quota exhaustion
    state = {"variant_idx": 0,
             "window_idx": 0,
             "order_idx": 0,
             "page": 0}

    # Completion flag:
    # 1 -> process completed correctly
    # 0 -> process interrupted by quota on last key
    completed = 1

    print(f"Keyword variants: {', '.join(variants)}")

    # Open NDJSON output (overwrite on each run)
    with open(output_json, "w", encoding="utf-8") as fout:

        # Iterate over available API keys
        for key_idx, api_key in enumerate(api_keys, 1):
            print(f"\nAPI KEY {key_idx}/{len(api_keys)}")

            yt = build("youtube", "v3", developerKey=api_key, cache_discovery=False)

            try:
                # Variant-level exploration (resume-aware)
                for v_idx in range(state["variant_idx"], len(variants)):
                    variant = variants[v_idx]
                    variant_norm = normalize_text(variant)
                    print(f"  Variant '{variant}'")

                    # Temporal window exploration (resume-aware)
                    for w_idx in range(state["window_idx"], len(windows)):
                        w_start, w_end = windows[w_idx]
                        print(f"    Window {w_idx + 1} of {len(windows)} "
                              f"from {w_start.date()} to {w_end.date()}")

                        # Ranking order exploration (resume-aware)
                        for o_idx in range(state["order_idx"], len(orders)):
                            order = orders[o_idx]

                            page_token = None
                            page_count = state["page"]

                            # Pagination loop
                            while page_count < max_pages_per_window:

                                # Retry loop for transient API failures
                                for attempt in range(1, max_retries + 1):
                                    try:
                                        req = yt.search().list(q=variant, part="snippet", type="video",
                                                               order=order,
                                                               publishedAfter=w_start.isoformat().replace("+00:00", "Z"),
                                                               publishedBefore=w_end.isoformat().replace("+00:00", "Z"),
                                                               maxResults=50, pageToken=page_token)
                                        res = req.execute(num_retries=0)
                                        break

                                    except HttpError as e:
                                        if e.resp.status == 403 and "quotaExceeded" in str(e):
                                            # Save resume state
                                            state.update({"variant_idx": v_idx,
                                                          "window_idx": w_idx,
                                                          "order_idx": o_idx,
                                                          "page": page_count})

                                            # Quota handling and logging
                                            if key_idx == len(api_keys):
                                                print(f"API KEY {key_idx} QUOTA EXCEEDED — PROCESS NOT COMPLETED")
                                                completed = 0
                                            else:
                                                print(f"API KEY {key_idx} QUOTA EXCEEDED — SWITCHING API KEY")

                                            raise StopIteration

                                        time.sleep(2 * attempt)

                                else:
                                    break

                                page_count += 1

                                video_ids = []
                                meta = {}

                                # Extract video identifiers and minimal snippet metadata
                                for item in res.get("items", []):
                                    vid = item["id"]["videoId"]
                                    snip = item["snippet"]

                                    video_ids.append(vid)
                                    meta[vid] = {"title": snip["title"],
                                                 "description": snip.get("description", ""),
                                                 "published_at": snip["publishedAt"],
                                                 "channel_id": snip["channelId"],
                                                 "channel_title": snip["channelTitle"]}

                                if not video_ids:
                                    break

                                # Fetch full snippets to access tags
                                vids_req = yt.videos().list(part="snippet", id=",".join(video_ids))
                                vids_res = vids_req.execute(num_retries=0)

                                # Keyword presence check
                                for v in vids_res.get("items", []):
                                    vid = v["id"]

                                    title_norm = normalize_text(meta[vid]["title"])
                                    tags_norm = [normalize_text(t) for t in v["snippet"].get("tags", [])]
                                    description_norm = normalize_text(meta[vid]["description"])

                                    from_title = variant_norm in title_norm
                                    from_tag = variant_norm in tags_norm
                                    from_desc = variant_norm in description_norm

                                    if from_title or from_tag or from_desc:
                                        event = {"video_id": vid,
                                                 "title": meta[vid]["title"],
                                                 "description": meta[vid]["description"],
                                                 "published_at": meta[vid]["published_at"],
                                                 "channel": {"id": meta[vid]["channel_id"],
                                                             "title": meta[vid]["channel_title"]},
                                                 "discovery": {"keyword": keyword,
                                                               "keyword_variant": variant,
                                                               "order": order,
                                                               "window": {"start": w_start.isoformat(),
                                                                          "end": w_end.isoformat()},
                                                               "matched_on": {"title": from_title,
                                                                              "tags": from_tag,
                                                                              "description": from_desc}},
                                                 "collected_at": datetime.now(UTC).isoformat(),
                                                 "api_key_used": key_idx}

                                        fout.write(json.dumps(event, ensure_ascii=False) + "\n")
                                        events.append(event)

                                page_token = res.get("nextPageToken")
                                time.sleep(sleep_time)

                                if not page_token:
                                    break

                            state["page"] = 0
                        state["order_idx"] = 0
                    state["window_idx"] = 0
                state["variant_idx"] = 0

                # SUCCESS: process completed correctly with this API key
                #completed = 1
                print(f"API KEY {key_idx} COMPLETED THE PROCESS")
                break

            except StopIteration:
                continue

    print(f"\n Total discovery events collected: {len(events)}")
    print(f"SAVED NDJSON written to: {output_json}")

    return events, completed


# VIDEO STATISTICS
# Function: youtube_video_statistics
# Retrieves engagement statistics and duration metadata for a set of
# YouTube videos previously discovered.
# The function processes video IDs in API-compliant batches, converts
# ISO 8601 durations into seconds, identifies short-form content,
# and handles quota limits through API key rotation with resume support.
# Writes raw statistics snapshots to a newline-delimited JSON (NDJSON) file.
#
# Input :
#   - api_keys (list[str])             available YouTube API keys
#   - video_ids (iterable)             collection of video IDs to enrich
#   - reel_threshold_seconds (int)     duration threshold for short-form content
#   - sleep_time (int)                 delay between API requests
#   - output_json (str)                output NDJSON file path
#   - max_retries (int)                retry attempts for transient failures
#
# Output:
#   - list[dict] → collected statistics snapshots
#   - int        → completion flag (1 = completed, 0 = quota interrupted)

def youtube_video_statistics(api_keys, video_ids,
                             reel_threshold_seconds=60,
                             sleep_time=2,
                             output_json=None,
                             max_retries=3):

    # Enforce explicit output path
    if output_json is None:
        raise ValueError("output_json must be provided explicitly")

    # Deduplicate video IDs to avoid redundant API calls
    video_ids = list(set(video_ids))
    events = []

    print(f" Total unique videos uploaded: {len(video_ids)}")

    # Handle empty input defensively
    if not video_ids:
        open(output_json, "w").close()
        return events, 1

    # Split video IDs into API-compliant batches (max 50 IDs per request)
    batches = list(chunk_list(video_ids, 50))

    # Resume state for batch-level processing
    state = {"batch_idx": 0}

    # Completion flag:
    # 1 -> process completed correctly
    # 0 -> process interrupted by quota on last key
    completed = 1

    # Open NDJSON output (overwrite on each run)
    with open(output_json, "w", encoding="utf-8") as fout:

        # Iterate over available API keys
        for key_idx, api_key in enumerate(api_keys, 1):
            print(f"\nAPI KEY {key_idx}/{len(api_keys)}")

            yt = build("youtube", "v3", developerKey=api_key, cache_discovery=False)

            try:
                # Batch-level enrichment loop (resume-aware)
                for b_idx in range(state["batch_idx"], len(batches)):
                    batch = batches[b_idx]

                    # Retry loop for transient API failures
                    for attempt in range(1, max_retries + 1):
                        try:
                            req = yt.videos().list(part="statistics,contentDetails",
                                                   id=",".join(batch))
                            res = req.execute(num_retries=0)
                            break

                        except HttpError as e:
                            if e.resp.status == 403 and "quotaExceeded" in str(e):
                                # Save resume state
                                state["batch_idx"] = b_idx

                                # Quota handling and logging
                                if key_idx == len(api_keys):
                                    print(f"API KEY {key_idx} QUOTA EXCEEDED — PROCESS NOT COMPLETED")
                                    completed = 0
                                else:
                                    print(f"API KEY {key_idx} QUOTA EXCEEDED — SWITCHING API KEY")

                                raise StopIteration

                            time.sleep(2 * attempt)

                        except (SocketTimeout, TimeoutError):
                            time.sleep(2 * attempt)

                    else:
                        # Skip batch if retries are exhausted
                        continue

                    # Timestamp for the current statistics snapshot
                    collected_at = datetime.now(UTC).isoformat()

                    # Parse statistics for each video in the batch
                    for item in res.get("items", []):
                        stats = item.get("statistics", {})
                        duration_iso = item["contentDetails"].get("duration")
                        duration_sec = parse_iso8601_duration(duration_iso)

                        event = {"video_id": item["id"],
                                 "statistics": {"view_count": int(stats.get("viewCount", 0)),
                                                "like_count": int(stats.get("likeCount", 0)),
                                                "comment_count": int(stats.get("commentCount", 0)),
                                                "favorite_count": int(stats.get("favoriteCount", 0)),
                                                "duration_seconds": duration_sec,
                                                "is_reel": (duration_sec is not None
                                                            and duration_sec < reel_threshold_seconds)},
                                 "collected_at": collected_at,
                                 "api_key_used": key_idx}

                        fout.write(json.dumps(event, ensure_ascii=False) + "\n")
                        events.append(event)

                    # Sleep between batches to avoid throttling
                    time.sleep(sleep_time)

                # SUCCESS: process completed correctly with this API key
                #completed = 1
                print(f"API KEY {key_idx} COMPLETED THE PROCESS")
                break

            except StopIteration:
                continue

    print(f"\n Enrichment snapshots collected: {len(events)}")
    print(f"SAVED NDJSON written to: {output_json}")

    return events, completed

# CHANNEL STATISTICS 
# Function: youtube_channel_statistics
# Fetches channel-level engagement statistics for a set of unique YouTube channels.
# Only numerical statistics are retrieved (no snippet, branding, or topic metadata).
# Data is written as raw snapshots to a newline-delimited JSON file.
#
# Input :
#   - api_keys (list[str])             available YouTube API keys
#   - channel_ids (iterable)           collection of channel IDs
#   - sleep_time (int)                 delay between API requests
#   - output_json (str)                output NDJSON file path
#   - max_retries (int)                retry attempts for transient failures
#
# Output:
#   - list[dict] → collected channel statistics snapshots
#   - int        → completion flag (1 = completed, 0 = quota interrupted)

def youtube_channel_statistics(api_keys, channel_ids,sleep_time=2,output_json=None,max_retries=3):

    # Enforce explicit output path
    if output_json is None:
        raise ValueError("output_json must be provided explicitly")

    # Deduplicate channel IDs to avoid redundant API calls
    channel_ids = list(set(channel_ids))
    events = []

    print(f" Total unique channels to fetch: {len(channel_ids)}")

    # Handle empty input defensively
    if not channel_ids:
        open(output_json, "w").close()
        return events, 1

    # Split channel IDs into API-compliant batches (max 50 IDs per request)
    batches = list(chunk_list(channel_ids, 50))

    # Completion flag:
    # 1 -> process completed correctly
    # 0 -> process interrupted by quota on last key
    completed = 1

    # Open NDJSON output (overwrite on each run)
    with open(output_json, "w", encoding="utf-8") as fout:

        # Iterate over available API keys
        for key_idx, api_key in enumerate(api_keys, 1):
            print(f"\nAPI KEY {key_idx}/{len(api_keys)}")

            yt = build("youtube", "v3",developerKey=api_key,cache_discovery=False)

            try:
                # Batch-level channel enrichment loop
                for batch_idx, batch in enumerate(batches):

                    # Retry loop for transient API failures
                    for attempt in range(1, max_retries + 1):
                        try:
                            req = yt.channels().list(part="statistics",
                                                     id=",".join(batch))
                            res = req.execute(num_retries=0)
                            break

                        except HttpError as e:
                            if e.resp.status == 403 and "quotaExceeded" in str(e):

                                # Quota handling and logging
                                if key_idx == len(api_keys):
                                    print(f"API KEY {key_idx} QUOTA EXCEEDED — PROCESS NOT COMPLETED")
                                    completed = 1
                                else:
                                    print(f"API KEY {key_idx} QUOTA EXCEEDED — SWITCHING API KEY")

                                raise StopIteration

                            time.sleep(2 * attempt)

                    else:
                        # Skip batch if retries are exhausted
                        continue

                    # Timestamp for the current channel snapshot
                    collected_at = datetime.now(UTC).isoformat()

                    # Parse numerical channel statistics
                    for ch in res.get("items", []):
                        stats = ch.get("statistics", {})

                        event = {"channel_id": ch["id"],
                                 "subscriber_count": (None if stats.get("hiddenSubscriberCount")
                                                      else int(stats.get("subscriberCount", 0))),
                                 "video_count": int(stats.get("videoCount", 0)),
                                 "view_count": int(stats.get("viewCount", 0)),
                                 "hidden_subscribers": bool(stats.get("hiddenSubscriberCount", False)),
                                 "collected_at": collected_at,
                                 "api_key_used": key_idx}

                        fout.write(json.dumps(event, ensure_ascii=False) + "\n")
                        events.append(event)

                    # Sleep between batches to avoid API throttling
                    time.sleep(sleep_time)

                # SUCCESS: process completed correctly with this API key
                #completed = 1
                print(f"API KEY {key_idx} COMPLETED THE PROCESS")
                break

            except StopIteration:
                continue

    print(f"\n Channel statistics collected: {len(events)}")
    print(f"SAVED NDJSON written to: {output_json}")

    return events, completed

# PIPELINE ORCHESTRATION
# Orchestrates the complete YouTube data acquisition workflow.
# This pipeline is intentionally limited to raw data collection and persistence.
# It defines the temporal scope of the acquisition, executes discovery and enrichment phases in sequence,
# and materializes raw outputs as NDJSON artifacts.
#
# Input :
#   - api_keys (list[str])             available YouTube API keys
#   - topic (str)                      topic keyword
#   - lookback_days (int)              temporal coverage in days
#   - window_days (int)                size of discovery sub-windows
#   - end_date (datetime | str | None) optional end date override
#   - output_dir (str | None)          directory for output artifacts
#
# Output:
#   - dict → paths to generated NDJSON artifacts and completion flags

def run_youtube_pipeline(api_keys,topic,lookback_days,window_days=7,end_date=None,output_dir=None):

    # Initialize output directory if provided
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    # Resolve end date
    if end_date is None:
        end_date = datetime.now(UTC)
    elif isinstance(end_date, str):
        end_date = datetime.fromisoformat(end_date.replace("Z", "+00:00"))

    # Compute start date from lookback window
    start_date = end_date - timedelta(days=lookback_days)

    # Log pipeline configuration
    print(f"Topic         : {topic}")
    print(f"Lookback days : {lookback_days}")
    print(f"Start date    : {start_date.date()}")
    print(f"End date      : {end_date.date()}")
    print(f"Window days   : {window_days}")

    print("\n Video discovery")

    # DISCOVERY PHASE
    discovery_file = make_output_name(topic, "discovery", output_dir)

    discovery_events, discovery_completed = youtube_search_videos(api_keys=api_keys,keyword=topic,
                                                                  start_date=start_date,end_date=end_date,
                                                                  window_days=window_days,output_json=discovery_file)

    total_events = len(discovery_events)
    unique_video_ids = {e["video_id"] for e in discovery_events}

    unique_count = len(unique_video_ids)
    duplicate_count = total_events - unique_count
    duplicate_pct = (duplicate_count / total_events * 100 if total_events > 0 else 0.0)

    print("\n Duplicate video analysis")
    print(f" Total discovery events : {total_events}")
    print(f" Unique video IDs       : {unique_count}")
    print(f" Duplicate videos       : {duplicate_count} ({duplicate_pct:.2f}% of total)")

    # Extract unique video IDs for statistics enrichment
    video_ids = list(unique_video_ids)

    print("\n Statistics and duration")

    # VIDEO STATISTICS PHASE
    stats_file = make_output_name(topic, "stats", output_dir)

    stats_events, stats_completed = youtube_video_statistics(api_keys=api_keys,video_ids=video_ids,
                                                             output_json=stats_file)

    print("\n Channel statistics")

    # CHANNEL STATISTICS PHASE
    channel_ids = list({e["channel"]["id"] for e in discovery_events if "channel" in e})

    channel_file = make_output_name(topic, "channel_stats", output_dir)

    channel_stats_events, channel_completed = youtube_channel_statistics(api_keys=api_keys,channel_ids=channel_ids,
                                                                         output_json=channel_file)

    # PIPELINE OUTPUT
    return {"discovery_file": discovery_file,
            "stats_file": stats_file,
            "channel_file": channel_file,
            "completed": {"discovery": discovery_completed,
                          "video_statistics": stats_completed,
                          "channel_statistics": channel_completed}
           }


## Google Trends

In [4]:
# GOOGLE TRENDS INGESTION
#
# Function: ingest_google_trends
# Ingests manually downloaded Google Trends CSV files (TOP and RISING queries), automatically discovering files 
# The function preserves all original CSV columns and enriches each dataset with pipeline metadata
# (keyword, temporal window, source file, ingestion time).
# Optionally exports unified TOP and RISING datasets as CSV files.
#
# Input :
#   - base_dir (str | Path)                                root directory containing Google Trends CSVs
#   - keyword (str)                                        seed keyword associated with the datasets
#   - export_dir (str | Path | None)                       optional output directory for unified CSV exports
#   - expected_end_date (datetime | None)                  optional expected end date used for consistency checks
#
# Output:
#   - dict{"top": pd.DataFrame, "rising": pd.DataFrame}    unified TOP and RISING Google Trends datasets

def ingest_google_trends(base_dir,keyword,export_dir=None,expected_end_date: datetime | None = None):

    # Resolve and validate the base directory containing Google Trends CSVs
    base_dir = Path(base_dir).expanduser().resolve()
    if not base_dir.exists():
        raise FileNotFoundError(f"Google Trends directory not found: {base_dir}")

    # Containers for TOP and RISING query DataFrames
    dfs_top = []
    dfs_rising = []

    # Filename pattern encoding query type, country, and temporal window
    pattern = re.compile(r"searched_with_(top|rising)-queries_([A-Z]{2})_(\d{8}-\d{4})_(\d{8}-\d{4})",
                         re.IGNORECASE,)

    # Recursively scan for CSV files
    for csv_file in base_dir.rglob("*.csv"):

        # Attempt to match the expected Google Trends filename pattern
        match = pattern.search(csv_file.name)
        if not match:
            # Skip files that do not conform to the naming convention
            continue

        # Extract metadata encoded in the filename
        query_type, country, start_raw, end_raw = match.groups()

        # Parse temporal window boundaries
        window_start = datetime.strptime(start_raw, "%Y%m%d-%H%M")
        window_end = datetime.strptime(end_raw, "%Y%m%d-%H%M")
        window_days = (window_end.date() - window_start.date()).days

        # Construct a logical identifier for the dataset
        logical_name = f"{query_type}_query_{keyword}_{window_days}d"

        # Load the CSV while preserving all original columns
        df = pd.read_csv(csv_file)

        # Enrich with pipeline metadata
        df["keyword"] = keyword
        df["query_type"] = query_type
        df["window_days"] = window_days
        df["window_start"] = window_start
        df["window_end"] = window_end
        df["logical_name"] = logical_name
        df["source_file"] = csv_file.name
        df["ingested_at"] = datetime.now(UTC)

        # Route the dataset to the appropriate container
        if query_type.lower() == "top":
            dfs_top.append(df)
        else:
            dfs_rising.append(df)

    # Concatenate all TOP and RISING datasets
    results = {"top": pd.concat(dfs_top, ignore_index=True) if dfs_top else pd.DataFrame(),
               "rising": pd.concat(dfs_rising, ignore_index=True) if dfs_rising else pd.DataFrame(),}

    # Data quality flags 
    trends_quality = {"top_end_date": None,"rising_end_date": None,}


    # Consistency check on temporal coverage
    if expected_end_date is not None:
        expected_date = expected_end_date.date()
    
        # Top check
        if not results["top"].empty:
            top_end_date = results["top"]["window_end"].max().date()
            trends_quality["top_end_date"] = (top_end_date == expected_date)
    
        # Rising check
        if not results["rising"].empty:
            rising_end_date = results["rising"]["window_end"].max().date()
            trends_quality["rising_end_date"] = (rising_end_date == expected_date)
    
        # Optional warning
        if (trends_quality["top_end_date"] is False or trends_quality["rising_end_date"] is False):
            print( "\nATTENTION: Google Trends temporal coverage mismatch.\n"
                    f"Expected end date : {expected_date}\n" +
                   (f"TOP end date mismatch\n" if trends_quality["top_end_date"] is False else "")+
                   (f"RISING end date mismatch\n" if trends_quality["rising_end_date"] is False else ""))

    # Optional export of unified datasets
    if export_dir:
        export_dir = Path(export_dir).expanduser().resolve()
        export_dir.mkdir(parents=True, exist_ok=True)

        kw_norm = normalize_keyword(keyword)

        results["top"].to_csv(export_dir / f"{kw_norm}_google_trends_top.csv",index=False,)

        results["rising"].to_csv(export_dir / f"{kw_norm}_google_trends_rising.csv",index=False,)

    return results, trends_quality

## SQLite storage

In [5]:
# LOAD YOUTUBE DISCOVERY NDJSON → DATAFRAME
#
# Function: load_youtube_discovery_ndjson_as_dataframe
# Loads raw YouTube discovery events stored in a newline-delimited JSON (NDJSON) file
# and converts them into a structured pandas DataFrame.
# The resulting DataFrame is compatible with the youtube_videos table schema and
# preserves discovery metadata such as keyword variants, temporal windows, and
# match origin (title, tags, description).
#
# Input :
#   - ndjson_path (str | Path)     path to the discovery NDJSON file
#
# Output:
#   - pd.DataFrame                structured discovery dataset

def load_youtube_discovery_ndjson_as_dataframe(ndjson_path: str | Path):

    # Resolve NDJSON path
    ndjson_path = Path(ndjson_path)

    # Handle missing input defensively
    if not ndjson_path.exists():
        print(f"Discovery NDJSON not found: {ndjson_path}")
        return pd.DataFrame()

    rows = []

    # Stream-read NDJSON file line by line
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            # Extract and flatten relevant discovery fields
            rows.append({"video_id": d["video_id"],
                         "title": d["title"],
                         "description": d.get("description", ""),
                         "published_at": d["published_at"],
                         "channel_id": d["channel"]["id"],
                         "channel_title": d["channel"]["title"],
                         "keyword": d["discovery"]["keyword"],
                         "keyword_variant": d["discovery"]["keyword_variant"],
                         "order": d["discovery"]["order"],
                         "window_start": d["discovery"]["window"]["start"],
                         "window_end": d["discovery"]["window"]["end"],
                         "from_title": int(d["discovery"]["matched_on"]["title"]),
                         "from_tag": int(d["discovery"]["matched_on"]["tags"]),
                         "from_description": int(d["discovery"]["matched_on"].get("description", 0)),
                         "collected_at": d["collected_at"],
                         "api_key_used": d["api_key_used"]})

    # Materialize DataFrame from collected rows
    df = pd.DataFrame(rows)
    before = len(df)

    df = df.drop_duplicates(subset=["video_id"], keep="first")

    after = len(df)

    print(f"Loaded discovery for {len(df)} YouTube videos from NDJSON")
    print(f"Removed {before - after} duplicate rows (same video_id)")

    return df


# LOAD YOUTUBE STATS NDJSON → DATAFRAME
#
# Function: load_youtube_stats_ndjson_as_dataframe
# Loads raw YouTube video statistics snapshots stored in NDJSON format and 
# converts them into a structured pandas DataFrame.
# Multiple snapshots per video may be present; only the most recent snapshot (by collected_at timestamp) is retained.
#
# Input :
#   - ndjson_path (str | Path)     path to the video statistics NDJSON file
#
# Output:
#   - pd.DataFrame                structured video statistics dataset

def load_youtube_stats_ndjson_as_dataframe(ndjson_path: str | Path):

    # Resolve NDJSON path
    ndjson_path = Path(ndjson_path)

    # Handle missing input defensively
    if not ndjson_path.exists():
        print("\n")
        print(f"Stats NDJSON not found: {ndjson_path}")
        return pd.DataFrame()

    rows = []

    # Stream-read NDJSON file line by line
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            s = d["statistics"]

            # Extract and flatten statistics fields
            rows.append({"video_id": d["video_id"],
                         "view_count": s["view_count"],
                         "like_count": s["like_count"],
                         "comment_count": s["comment_count"],
                         "favorite_count": s["favorite_count"],
                         "duration_seconds": s["duration_seconds"],
                         "is_reel": int(s["is_reel"]),
                         "collected_at": d["collected_at"]})

    # Materialize DataFrame from collected rows
    df = pd.DataFrame(rows)

    # Retain the most recent snapshot per video
    df = (df.sort_values("collected_at")
            .drop_duplicates(subset="video_id", keep="last")
            .reset_index(drop=True))
    print(f"Loaded statistics for {len(df)} YouTube videos from NDJSON")

    return df


# LOAD YOUTUBE CHANNEL STATS NDJSON → DATAFRAME
#
# Function: load_youtube_channel_stats_ndjson_as_dataframe
# Loads raw YouTube channel-level statistics snapshots stored in NDJSON format
# and converts them into a structured pandas DataFrame.
# If multiple snapshots are present for the same channel, only the most recent
# snapshot is retained.
#
# Input :
#   - ndjson_path (str | Path)     path to the channel statistics NDJSON file
#
# Output:
#   - pd.DataFrame                structured channel statistics dataset

def load_youtube_channel_stats_ndjson_as_dataframe(ndjson_path: str | Path):

    # Resolve NDJSON path
    ndjson_path = Path(ndjson_path)

    # Handle missing input defensively
    if not ndjson_path.exists():
        print(f"Channel statistics NDJSON not found: {ndjson_path}")
        return pd.DataFrame()

    rows = []

    # Stream-read NDJSON file line by line
    with open(ndjson_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)

            # Extract channel-level statistics fields
            rows.append({"channel_id": d["channel_id"],
                         "subscriber_count": d["subscriber_count"],
                         "video_count": d["video_count"],
                         "view_count": d["view_count"],
                         "hidden_subscribers": int(d["hidden_subscribers"]),
                         "collected_at": d["collected_at"]})

    # Materialize DataFrame from collected rows
    df = pd.DataFrame(rows)

    # Retain the most recent snapshot per channel
    df = (df.sort_values("collected_at")
            .drop_duplicates(subset="channel_id", keep="last")
            .reset_index(drop=True))

    print(f"Loaded statistics for {len(df)} YouTube channels from NDJSON")

    return df


In [6]:
# SQLITE UTILITIES
#
# This module provides low-level helpers for managing a SQLite database used as storage backend for the pipeline.
# Responsibilities are intentionally limited to:
#   - destructive maintenance operations (database wipe)
#   - ingestion of pre-materialized pandas DataFrame

# Completely wipe a SQLite database by dropping all tables (and optionally views).
# This utility is intended for controlled resets of the storage layer during
# development, experimentation, or full pipeline re-runs.
#
# Parameters
#   - db_path (str | Path)     path to the SQLite database file
#   - drop_views (bool)        if True, also drop all SQL views

def wipe_sqlite_database(db_path: str | Path, drop_views: bool = True):
    # Resolve database path
    db_path = Path(db_path)

    # Handle missing database defensively
    if not db_path.exists():
        #print("SQLite DB does not exist, nothing to wipe.")
        return

    # Open a connection to the SQLite database
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        # Retrieve all table names from the SQLite catalog
        cursor.execute("""SELECT name FROM sqlite_master
                          WHERE type='table'""")
        tables = cursor.fetchall()

        # Drop each table explicitly
        for (table_name,) in tables:
            cursor.execute(f"DROP TABLE IF EXISTS {table_name}")

        # Optionally drop all SQL views
        if drop_views:
            cursor.execute("""SELECT name FROM sqlite_master
                              WHERE type='view'""")
            views = cursor.fetchall()

            for (view_name,) in views:
                print(f"Dropping view: {view_name}")
                cursor.execute(f"DROP VIEW IF EXISTS {view_name}")

    print("SQLite database wiped successfully.")


# Ingest a pandas DataFrame into a SQLite table using a parameterized schema.
# This function is intentionally domain-agnostic and operates exclusively
# on pre-materialized pandas DataFrames.
# It assumes that all parsing, validation, and schema alignment steps
# have already been performed upstream.
#
# The ingestion workflow is explicitly divided into two phases:
#   1) DDL phase: drop and recreate the target table using a parameterized
#      CREATE TABLE statement (topic-first schema)
#   2) DML phase: bulk insert of rows via pandas.to_sql
#
# Parameters
#   - df (pd.DataFrame)                pre-materialized dataset to ingest
#   - table_name (str)                 target SQLite table name
#   - create_table_sql (str)           CREATE TABLE template with {table} placeholder
#   - sqlite_db_path (str | Path)      path to the SQLite database file

def ingest_dataframe_sqlite(df, table_name, create_table_sql, sqlite_db_path):

    # Skip ingestion entirely if the DataFrame is empty or undefined.
    # In this case, the target table is not created to avoid
    # polluting the database with empty or meaningless artifacts.
    if df is None or df.empty:
        print(f" Skipping {table_name}: empty DataFrame")
        return

    # Resolve database path and ensure that the parent directory exists
    sqlite_db_path = Path(sqlite_db_path)
    sqlite_db_path.parent.mkdir(parents=True, exist_ok=True)

    # DDL PHASE
    # Explicitly reset the target table schema by dropping any existing table
    # and recreating it using the provided parameterized CREATE TABLE statement.
    with sqlite3.connect(sqlite_db_path) as conn:
        cursor = conn.cursor()

        cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        cursor.execute(create_table_sql.format(table=table_name))
        conn.commit()

    # DML PHASE
    # Bulk insert DataFrame rows into the freshly created table.
    # pandas.to_sql is used here for simplicity, correctness, and reliability.
    with sqlite3.connect(sqlite_db_path) as conn:
        df.to_sql(name=table_name,con=conn,if_exists="append",index=False)

    print(f"Ingested {len(df)} rows into '{table_name}'")


In [7]:
# RELATIONAL SCHEMA
#
# The following CREATE TABLE templates define the relational schema
# used by the Content Engine pipeline to persist all collected data
# in a SQLite database using a topic-first strategy.
#
# Each table is instantiated with a topic-specific prefix
# (e.g. data_science_youtube_videos), ensuring full isolation
# between pipeline runs and preventing cross-topic contamination.
#
# Integrated data sources:
#   - YouTube Data API:
#        video discovery metadata (titles, channels, keyword matches)
#        engagement statistics and duration information
#   - Google Trends:
#        related top queries
#        related rising queries
#
# Design principles:
# - Topic-scoped relational tables (no global base tables)
# - Clear separation between discovery data and enrichment data
# - Explicit column typing for SQL analytics and reproducibility
# - Boolean semantic flags stored as INTEGER with CHECK constraints
# - Primary keys to enforce entity-level uniqueness
#
# These schemas are intended to be used as parameterized templates and must be formatted with
# a concrete table name before execution

# YOUTUBE — VIDEO DISCOVERY
CREATE_YOUTUBE_VIDEOS = """
CREATE TABLE {table} (
    video_id TEXT PRIMARY KEY,
    title TEXT,
    description TEXT,
    published_at TEXT,
    channel_id TEXT,
    channel_title TEXT,

    keyword TEXT,
    keyword_variant TEXT,
    "order" TEXT,

    window_start TEXT,
    window_end TEXT,

    from_title INTEGER NOT NULL CHECK (from_title IN (0, 1)),
    from_tag INTEGER NOT NULL CHECK (from_tag IN (0, 1)),
    from_description INTEGER NOT NULL CHECK (from_description IN (0, 1)),

    collected_at TEXT,
    api_key_used INTEGER
);
"""

# YOUTUBE — VIDEO STATISTICS and DURATION
CREATE_YOUTUBE_VIDEO_STATS = """
CREATE TABLE {table} (
    video_id TEXT PRIMARY KEY,
    view_count INTEGER,
    like_count INTEGER,
    comment_count INTEGER,
    favorite_count INTEGER,
    duration_seconds INTEGER,

    is_reel INTEGER NOT NULL CHECK (is_reel IN (0, 1)),

    collected_at TEXT
);
"""

# YOUTUBE — CHANNEL STATISTICS
CREATE_YOUTUBE_CHANNEL_STATS = """
CREATE TABLE {table} (
    channel_id TEXT PRIMARY KEY,
    subscriber_count INTEGER,
    video_count INTEGER,
    view_count INTEGER,

    hidden_subscribers INTEGER NOT NULL CHECK (hidden_subscribers IN (0, 1)),

    collected_at TEXT
);
"""

# GOOGLE TRENDS — TOP RELATED QUERIES
CREATE_GOOGLE_TRENDS_TOP = """
CREATE TABLE {table} (
    query TEXT NOT NULL,
    search_interest INTEGER,
    increase_percent TEXT,

    keyword TEXT NOT NULL,
    query_type TEXT NOT NULL,
    window_days INTEGER NOT NULL,
    window_start TEXT NOT NULL,
    window_end TEXT NOT NULL,

    logical_name TEXT,
    source_file TEXT,
    ingested_at TEXT,

    PRIMARY KEY (query, keyword, window_days)
);
"""

# GOOGLE TRENDS — RISING RELATED QUERIES
CREATE_GOOGLE_TRENDS_RISING = """
CREATE TABLE {table} (
    query TEXT NOT NULL,
    search_interest INTEGER,
    increase_percent TEXT,

    keyword TEXT NOT NULL,
    query_type TEXT NOT NULL,
    window_days INTEGER NOT NULL,
    window_start TEXT NOT NULL,
    window_end TEXT NOT NULL,

    logical_name TEXT,
    source_file TEXT,
    ingested_at TEXT,

    PRIMARY KEY (query, keyword, window_days)
);
"""

## Full pipeline

In [1]:
def run_full_acquisition_and_storage(youtube_api_key_file,youtube_topic,youtube_lookback_days,youtube_window_days,
                                     trends_keyword,trends_csv_root,sqlite_db_path,output_dir,youtube_end_date=None,
                                     wipe_db=False,):

    # Output directory initialization
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    start_time = datetime.now(UTC)

    # Youtube acquisition
    # Load and validate YouTube API keys (supports key rotation)
    api_keys = load_youtube_api_keys(youtube_api_key_file)

    # Execute the YouTube acquisition pipeline
    yt_results = run_youtube_pipeline(api_keys=api_keys,
                                      topic=youtube_topic,
                                      lookback_days=youtube_lookback_days,
                                      window_days=youtube_window_days,
                                      end_date=youtube_end_date,
                                      output_dir=output_dir,)

    # Extract NDJSON artifact paths from pipeline output
    discovery_file = yt_results["discovery_file"]
    stats_file     = yt_results["stats_file"]
    channel_file   = yt_results["channel_file"]

    # Extract completion flags (IMPORTANT)
    yt_completed = yt_results.get("completed", {})

    # Transform NDJSON into DataFrame
    # Load discovery events
    df_videos = load_youtube_discovery_ndjson_as_dataframe(discovery_file)

    # Deduplication of videos
    df_videos = (df_videos.sort_values("collected_at")
                          .drop_duplicates(subset="video_id", keep="first")
                          .reset_index(drop=True))

    # Load video-level statistics
    df_stats = load_youtube_stats_ndjson_as_dataframe(stats_file)

    # Load channel-level statistics
    df_channels = load_youtube_channel_stats_ndjson_as_dataframe(channel_file)

    print("\n          YouTube video, statistics and channel information acquired\n")
    end_time = datetime.now(UTC)
    print(f"Concluded in {end_time-start_time}")

    # Google Trends ingestion
    print("\n          Google trends acquisition\n")
    start_time = datetime.now(UTC)

    trends_results, trends_quality = ingest_google_trends(base_dir=trends_csv_root,keyword=trends_keyword,
                                                          export_dir=output_dir,expected_end_date=youtube_end_date)

    df_trends_top    = trends_results["top"]
    df_trends_rising = trends_results["rising"]

    df_trends_top = df_trends_top.rename(columns={"search interest": "search_interest",
                                                  "increase percent": "increase_percent"})

    df_trends_rising = df_trends_rising.rename(columns={"search interest": "search_interest",
                                                        "increase percent": "increase_percent"})

    print("\n          Google Trends acquisition completed\n")
    end_time = datetime.now(UTC)
    print(f"Concluded in {end_time-start_time}")
    
    # SQLITE STORAGE
    print("\n          Storing into SQLite\n")
    start_time = datetime.now(UTC)

    # Optional destructive reset of the entire database.
    # This should be enabled only for controlled full re-runs.
    if wipe_db:
        wipe_sqlite_database(sqlite_db_path, drop_views=True)

    # Normalize topic name to generate topic-prefixed table names
    kw = normalize_keyword(youtube_topic)

    # Ingest YouTube discovery metadata
    ingest_dataframe_sqlite(df_videos,f"{kw}_youtube_videos",CREATE_YOUTUBE_VIDEOS,sqlite_db_path)

    # Ingest YouTube video statistics
    ingest_dataframe_sqlite(df_stats,f"{kw}_youtube_video_stats",CREATE_YOUTUBE_VIDEO_STATS,sqlite_db_path)

    # Ingest YouTube channel statistics
    ingest_dataframe_sqlite(df_channels,f"{kw}_youtube_channel_stats",CREATE_YOUTUBE_CHANNEL_STATS,sqlite_db_path)

    # Ingest Google Trends TOP queries
    ingest_dataframe_sqlite(df_trends_top,f"{kw}_google_trends_top",CREATE_GOOGLE_TRENDS_TOP,sqlite_db_path)

    # Ingest Google Trends RISING queries
    ingest_dataframe_sqlite(df_trends_rising,f"{kw}_google_trends_rising",CREATE_GOOGLE_TRENDS_RISING,sqlite_db_path)

    end_time = datetime.now(UTC)
    print("\n          Storing completed\n")
    print(f"Concluded in {end_time-start_time}")

    return {"counts": {"youtube_videos": len(df_videos),
                       "youtube_stats": len(df_stats),
                       "youtube_channels": len(df_channels),
                       "google_trends_top": len(df_trends_top),
                       "google_trends_rising": len(df_trends_rising),},
            "completed": yt_completed,
            "google_top_date": trends_quality["top_end_date"],
            "google_rising_date": trends_quality["rising_end_date"],}